# LangChain MCP 클라이언트 연동

MCP 서버의 도구를 LangChain 도구로 변환해 기존 모델 호출 코드에서 사용하려면 어떤 어댑터가 필요할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. 비밀 값은 출력하지 않습니다.

## 패키지 버전 확인

패키지 버전 확인에 필요한 값과 결과를 확인합니다.

In [1]:
# 패키지 버전 확인에 필요한 값을 준비하고 결과를 확인합니다.
%pip install -q "langchain[mcp]==1.4.0" langchain-openai==1.6.0 fastmcp==4.0.3 mcp==2.2.0 python-dotenv==1.2.3
import warnings
from importlib.metadata import version
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", message=r".*langchain\.mcp.*", category=LangChainBetaWarning)

for package in ["langchain", "langchain-openai", "fastmcp", "mcp"]:
    print(package, version(package))

Note: you may need to restart the kernel to use updated packages.


langchain 1.4.0
langchain-openai 1.6.0
fastmcp 4.0.3
mcp 2.2.0


## MCPAdapter로 stdio 도구 발견

MCPAdapter로 stdio 도구 발견에 필요한 값과 결과를 확인합니다.

In [2]:
# MCPAdapter로 stdio 도구 발견에 필요한 값을 준비하고 결과를 확인합니다.
from pathlib import Path
from langchain.mcp import MCPAdapter

relative = Path("labs/part6/ch28-mcp/course_server.py")
server_path = next(folder / relative for folder in [Path.cwd(), *Path.cwd().parents] if (folder / relative).is_file())

async def load_and_call() -> tuple[list, dict, object]:
    async with MCPAdapter(server_path) as adapter:
        tools = await adapter.list_tools()
        raw_schema = tools[0].args_schema
        schema = raw_schema if isinstance(raw_schema, dict) else raw_schema.model_json_schema()
        result = await tools[0].ainvoke({"topic": "운영"})
        return tools, schema, result

tools, schema, result = await load_and_call()
print("tool:", tools[0].name)
print("required:", schema.get("required"))
print("result:", result)
assert tools[0].name == "lookup_policy"
assert schema.get("required") == ["topic"]
assert "20:00" in str(result)

tool: lookup_policy
required: ['topic']
result: [{'type': 'text', 'text': '평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.', 'id': 'lc_1fcdef69-62fe-406a-ac93-29caa83fbf75'}]


## 직접 LangChain 도구와 인터페이스 비교

직접 LangChain 도구와 인터페이스 비교에 필요한 값과 결과를 확인합니다.

In [3]:
# 직접 LangChain 도구와 인터페이스 비교에 필요한 값을 준비하고 결과를 확인합니다.
from langchain.tools import tool

@tool
def direct_lookup_policy(topic: str) -> str:
    """자료실 규정을 직접 조회합니다."""
    return {"운영": "평일 09:00~20:00"}[topic]

comparison = {
    "direct": (direct_lookup_policy.name, bool(direct_lookup_policy.args_schema)),
    "mcp": (tools[0].name, bool(tools[0].args_schema)),
}
print(comparison)
print("direct result:", direct_lookup_policy.invoke({"topic": "운영"}))
print("mcp result:", result)
assert all(has_schema for _, has_schema in comparison.values())

{'direct': ('direct_lookup_policy', True), 'mcp': ('lookup_policy', True)}
direct result: 평일 09:00~20:00
mcp result: [{'type': 'text', 'text': '평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.', 'id': 'lc_1fcdef69-62fe-406a-ac93-29caa83fbf75'}]
